# Classification examples

In [1]:
from sklearn.datasets import load_wine
import itertools
import sys

sys.path.append("..")

from ai_toolkit import (
    BaseDataset,
    MlTrainerConfig,
    LogisticRegressionModel,
    NaiveBayesModel,
    XGBoostModel,
    get_all_classification_models, 
    ClassificationModelTrainer, 
    lazypredict_classification,
    EnsembleVotingClassifierModel,
    EnsembleStackingClassifierModel,
    display_banner,
)

In [2]:
display_banner()

    _    ___           _____ ___   ___  _     _  _____ _____ 
   / \  |_ _|         |_   _/ _ \ / _ \| |   | |/ /_ _|_   _|
  / _ \  | |   _____    | || | | | | | | |   | ' / | |  | |  
 / ___ \ | |  |_____|   | || |_| | |_| | |___| . \ | |  | |  
/_/   \_\___|           |_| \___/ \___/|_____|_|\_\___| |_|  
                                                             

Version: 0.1.0
Author:  Chrissi


## Example data

In [2]:
class ClfDataset(BaseDataset):
    """Dataset for wine multi-class classification task.
    https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html
    """

    def __init__(self):
        """Initialize the clf dataset."""

        super().__init__()

    def load_data(self):
        """Load the wine dataset for classification task."""
        
        self.X, self.y = load_wine(return_X_y=True, as_frame=True)
        self.X_test = self.X.head()

In [3]:
CDataset = ClfDataset()
CDataset.load_data()
CDataset.preprocess()
X_clf, y_clf, X_test_clf = CDataset.get_data()

## Configuration

In [ ]:
TrainerConfig = MlTrainerConfig()
TrainerConfig.N_TRAILS = 2

## Training and evaluation

### Train one example model

In [5]:
base_model = LogisticRegressionModel()

# Create a classification model trainer
trainer = ClassificationModelTrainer(
    base_model=base_model,
    config=TrainerConfig,
)

In [ ]:
# Train and optimize the model
best_model, mean_metrics = trainer.train_and_optimize(
    X=X_clf, 
    y=y_clf, 
    n_trials=TrainerConfig.N_TRAILS,
)

# y_pred, y_pred_proba = trainer.predict(X_test)

### Train all classification models

In [ ]:
base_models = get_all_classification_models()

for base_model in base_models.values():

    # Create a classification model trainer
    trainer = ClassificationModelTrainer(
        base_model=base_model,
        config=TrainerConfig,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_clf, 
        y=y_clf, 
        n_trials=TrainerConfig.N_TRAILS,
    )

In [ ]:
df_mean_results = lazypredict_classification(
    X=X_clf, 
    y=y_clf, 
    n_splits=TrainerConfig.N_SPLITS,
    random_state=TrainerConfig.RANDOM_STATE,
)

# print(df_mean_results.to_string())
df_mean_results

### Ensemble

In [ ]:
# (Model, mlflow run_id) pairs
model_pool = [
    (LogisticRegressionModel(), "0f1ed6332d7546a8b40fb8bdd1f3176c"),
    (NaiveBayesModel(), "54e32a7a289d46488ae01625119ff369"),
    (XGBoostModel(), "40e33107c2274c969e99e5d09bf786aa"),
]

meta_model = LogisticRegressionModel()

In [ ]:
combinations = []
 
for model in range(2, len(model_pool) + 1):
    combinations.extend(itertools.combinations(model_pool, model))

#### Voting | Train all combinations

In [ ]:
for combination in combinations:
    models = list(combination)

    base_model = EnsembleVotingClassifierModel(
        models=models,
    )

    # Create a classification model trainer
    trainer = ClassificationModelTrainer(
        base_model=base_model,
        config=TrainerConfig,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_clf, 
        y=y_clf, 
        n_trials=TrainerConfig.N_TRAILS,
    )

    # y_pred, y_pred_proba = trainer.predict(X_test)

#### Stacking | Train all combinations

In [ ]:
for combination in combinations:
    models = list(combination)
    
    base_model = EnsembleStackingClassifierModel(
        models=models,
        meta_model=meta_model,
    )

    # Create a classification model trainer
    trainer = ClassificationModelTrainer(
        base_model=base_model,
        config=TrainerConfig,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_clf, 
        y=y_clf, 
        n_trials=TrainerConfig.N_TRAILS,
    )

    # y_pred, y_pred_proba = trainer.predict(X_test)